## 9.4 RECURSIVIDAD. BACKTRACKING

La estrategia conocida como backtracking consiste en ir tomando decisiones que nos lleven por diferentes caminos para intentar (aplicando recursión) llegar a una solución. Cuando se llega a punto en que ya es solución se termina, pero en cuanto lleguemos a un punto en que ya sabemos que no es solución, entonces volvemos **hacia atrás** (_backtrack_) hacia donde tomamos la decisión y deshacemos lo que hicimos e intentar por otro camino.

### 9.4.1 Dada una lista de números positivos encontrar una secuencia cuya suma sea igual a un valor objetivo
Note que en este caso puede no haber solución porque no hay ninguna secuencia cuya sume es igual al valor objetivo o pueden haber varias porque hay mas de una secuencia que alcancen esa suma.
Tenemos una lista de números enteros positivos (mayores o iguales que cero) queremos encontrar una sublista cuya suma sea igual a un cierto _valor_objetivo_. La estrategia consiste en ir recorriendo la lista e ir formando una secuencia hasta que la suma sea igual al valor que se quiere alcanzar o hasta que la suma sobrepase este valor lo que implica que debemos descartar esa secuencia. Para descartar una secuencia quitamos al último elemento añadido a la secuencia y probamos con otro de la secuencia al añadir el elemento en la posición i+1 de la lista original. Si agotamos todos los posibles caminos intentando empezar por cualquiera de los elementos de la secuencia entonces se retorna falso porque no hay solución



In [ ]:
def buscar_sec_suma(nums, valor_objetivo):
    #Se supone nums una lista de números positivos o 0 y objetivos un entero >= 0
    def backtrack(i, suma_actual, secuencia):
        # estamos probando con secuencias que empiezan en el valor en la posición i de la lista nums
        if suma_actual == valor_objetivo and len(secuencia) > 0: #Tenemos una secuencia cuya suma da el objetivo
            return True, secuencia #Note que la primera llamada a la recursión no puede salir por aquí porque secuencia es la lista vacía [] por lo que su longitud no puede ser > 0

        # Poda: imposible continuar por este porque sobrepasamos el valor objetivo o ya no quedan más elementos de la lista nums por probar
        if suma_actual > valor_objetivo or i == len(nums):
            return False,[]

        # Rama 1: Explorar un nuevo camino añadiendo a la secuencia el elemento en la posición i
        secuencia.append(nums[i])
        hay_camino, resultado = backtrack(i + 1, suma_actual + nums[i], secuencia)
        if hay_camino:
            return True,resultado
        secuencia.pop() #Si no llego a ningún camino con este elemento en la secuencia lo quitamos

        # Rama 2: Obviamos ese elemento y exploramos un posible camino que incluya el siguiente
        return backtrack(i + 1, suma_actual, secuencia)
        if hay_camino:
            return True, resultado
        hay_camino, resultado = backtrack(i + 1, suma_actual, secuencia)
        if hay_camino:
            return True, resultado

        return False, [] #empezando en ese indice no hay ningún camimo, vuelta atras con false

    return backtrack(0, 0, []) #La primera llamada quiere decir prueba con el elemento en la posición 0 de la lista original, el valor en suma actual es 0 y la secuencia es la lista vacía vacia

print(buscar_sec_suma([1, 2, 3, 4, 5], 5))
print(buscar_sec_suma([1,8,4,2,7,3], 6))
print(buscar_sec_suma([1,2], 10))
print(buscar_sec_suma([0,4,2,7,3], 6))
print(buscar_sec_suma([1,1,2], 0))
print(buscar_sec_suma([0,0], 0))
print(buscar_sec_suma([1,2,4,7,5], 9))
print(buscar_sec_suma([], 0))


(True, [1, 4])
(True, [1, 2, 3])
(False, [])
(True, [0, 4, 2])
(False, [])
(True, [0])
(True, [2, 7])
(False, [])


#### Problema
Note que la solución anterior da la primera secuencia que encuentra y que cuya suma es igual al valor que se quiere alcanzar y da `False` si no exite tal lista. Pero, pueden haber otras secuencias que den también esa suma. Modifique el código anterior para dar la secuencia de menor logitud cuya suma sea igual al valor que se quiere alcanzar. Por ejemplo, para la llamada `buscar_sec_suma([1,8,4,2,7,3], 6``)` la respuesta debiera ser `(True, [4,2])` y no `(True, [1,2,3])` que es la respuesta que da el código anterior

### 9.4.2 Colocar 8 reinas en un tablero de ajedrez sin que se amenacen entre ellas

El problema consiste en encontrar una configuración de un tablero de ajedrez en la que se puedan poner 8 reinas  sin que se amenacen entre ellas. Para representarnos las distintas posiciones de un tablero usaremos una lista de longitud 8, las posiciones del 0 al 7 de esa lista nos representarán las 8 filas del tablero (visualicelo de arriba hacia abajo). Los elementos de la lista pueden ser valores entre -1 y 7. Un valor -1 en la posición i de la lista significa que en la fila i no hay ninguna reina, un valor j (0<=j<=7) significa que en la columna j de la fila i hay una reina.

Por ejemplo, la lista `[0, -1, -1, 6, -1, -1, -1, -1]` indica que hay una reina en la casilla `0,0` del tablero y una reina en casilla `3,6` (que sería en la columna `7` de la fila `4` del tablero)


 Q  .  .  .  .  .  .  .
 .  .  .  .  .  .  .  .
 .  .  .  .  .  .  .  .
 .  .  .  .  .  .  Q  .
 .  .  .  .  .  .  .  .
 .  .  .  .  .  .  .  .
 .  .  .  .  .  .  .  .
 .  .  .  .  .  .  .  .

El código a continuación empieza probando poner una reina en la columna 0 de la fila 0 (casilla (0,0)), como sabemos que una solución no puede tener dos reinas en una misma fila intenta poner luego una reina en alguna de las columnas de la fila 1 (casillas (1,_)) de modo que no amenace las ya puestas. Si esto es posible entonces intenta recursivamente
poner las reinas restantes en las filas restantes. Si por este camino no llega a una solución entonces vuelve atrás (backtrack) para probar en otra posición de columna dentro de la misma fila y si a su vez no encuentra solución y ya ha agotado todas las posiciones en esa fila entonces a su vez vuelve atrás para intentar en otra posición de la fila anterior.

Si se agotaran todas las posiciones entonces el problema no tiene solución, aunque sabemos que para este problema particular si la tiene. La función no_hay_amenazas(tablero, fila, columna) comprueba si poniendo una reina en la casilla (fila,columna) no se amenazan las reinas ubicadas anteriormente.



In [4]:
def se_amenazan(tablero, fila, col):
    for i in range(fila):
        if tablero[i][col]:
            return True
    i, j = fila, col
    while i >= 0 and j >= 0:
        if tablero[i][j]:
            return True
        i -= 1
        j -= 1
    i, j = fila, col
    while i >= 0 and j < 8:
        if tablero[i][j]:
            return True
        i -= 1
        j += 1
    return False

def ubica_reinas(tablero, fila):
    if fila == len(tablero):
        return True #ya ubicamos una reina en cada fila
    for col in range(len(tablero)):
        if not se_amenazan(tablero, fila, col):
            tablero[fila][col] = True
            # print("....") #descomentar estos print si queremos ir viendo como se imprimen los tableros
            # print_tablero(tablero)
            if ubica_reinas(tablero, fila + 1):
                return True
            tablero[fila][col] = False
            # print()
            # print_tablero(tablero)
    return False

def print_tablero(tablero):
    for i in range(len(tablero)):
        linea = ""
        for j in range(len(tablero)):
            if tablero[i][j]:
                linea += " Q "
            else:
                linea += " * "
        print(linea)

n = 8 #tamaño de tablero
tablero = [[False]*n for _ in range(n)]
print("\nTABLERO ORIGINAL")
print_tablero(tablero)
ubica_reinas(tablero, 0)
print('\nTABLERO FINAL')
print_tablero(tablero)




TABLERO ORIGINAL
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 
 *  *  *  *  *  *  *  * 

TABLERO FINAL
 Q  *  *  *  *  *  *  * 
 *  *  *  *  Q  *  *  * 
 *  *  *  *  *  *  *  Q 
 *  *  *  *  *  Q  *  * 
 *  *  Q  *  *  *  *  * 
 *  *  *  *  *  *  Q  * 
 *  Q  *  *  *  *  *  * 
 *  *  *  Q  *  *  *  * 
